# DEH 30-Day PySpark Challenge
## Day 7 — Filtering, Literal & Casting

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

orders_df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print(f"Rows: {orders_df.count()}")

## Step 5 — Basic filtering

In [ ]:
# filter() and where() are identical
high_value = orders_df.filter(F.col("unit_price") > 500)
print(f"Orders over $500: {high_value.count()}")

# SQL string syntax
orders_df.filter("status = 'Delivered'").show(3)

## Step 6 — Multiple conditions

In [ ]:
# AND — use & operator, wrap conditions in parentheses
orders_df.filter(
    (F.col("unit_price") > 500) &
    (F.col("status") == "Delivered")
).show(3)

# OR — use | operator
orders_df.filter(
    (F.col("region") == "East") |
    (F.col("region") == "West")
).show(3)

# NOT — use ~ operator
orders_df.filter(
    ~(F.col("status") == "Cancelled")
).show(3)

## Step 7 — Useful filter methods

In [ ]:
# isin() — match against a list
orders_df.filter(F.col("region").isin("East", "West")).show(3)

# between() — inclusive range
orders_df.filter(F.col("unit_price").between(100, 500)).show(3)

# contains() — string match
orders_df.filter(F.col("status").contains("liver")).show(3)

# isNotNull()
orders_df.filter(F.col("region").isNotNull()).show(3)

## Step 8 — lit() — constant column values

In [ ]:
# Add constant columns using lit()
orders_df = orders_df \
    .withColumn("source_system",    F.lit("orders_api")) \
    .withColumn("pipeline_version", F.lit(1)) \
    .withColumn("is_processed",     F.lit(True))

orders_df.select(
    "order_id", "source_system", "pipeline_version", "is_processed"
).show(3)

## Step 9 — cast() — converting column types

In [ ]:
# Read without schema — everything is string
orders_raw = spark.read \
    .option("header", "true") \
    .csv(f"s3a://{BUCKET}/data/orders.csv")

print("Before cast:")
orders_raw.select("quantity", "unit_price").printSchema()

# Cast using string type names
orders_casted = orders_raw \
    .withColumn("quantity",     F.col("quantity").cast("integer")) \
    .withColumn("unit_price",   F.col("unit_price").cast("double")) \
    .withColumn("discount_pct", F.col("discount_pct").cast("integer")) \
    .withColumn("order_date",   F.col("order_date").cast("date"))

print("After cast:")
orders_casted.select("quantity", "unit_price", "discount_pct", "order_date").printSchema()

In [ ]:
# What happens when cast fails — null, no exception
from pyspark.sql import Row

test_df = spark.createDataFrame([
    Row(val="123"),
    Row(val="456"),
    Row(val="not_a_number"),
    Row(val="789")
])

test_df.withColumn("val_int", F.col("val").cast("integer")).show()
# not_a_number becomes null — no error

---
## Assignment — Day 7

---

### Task 1
Filter `orders.csv` to show only orders where `unit_price` > 200 AND `status` is 'Delivered'.  
How many rows match?  
Then filter for orders where `region` is either 'East' OR 'West'.

In [ ]:
# Task 1 — Write your code here


### Task 2
Using `isin()`, filter orders where `payment_method` is 'Credit Card' or 'PayPal'.  
Then use `between()` to filter orders where `unit_price` is between 50 and 300.

In [ ]:
# Task 2 — Write your code here


### Task 3
Add three constant columns to `orders.csv` using `lit()`:  
- `data_source` = "DEH_CHALLENGE"
- `version` = 1  
- `is_active` = True

Show `order_id` and all three new columns.

In [ ]:
# Task 3 — Write your code here


### Task 4
Read `orders.csv` without a schema.  
Use `cast()` to convert `quantity` to integer, `unit_price` to double, and `order_date` to date.  
Print the schema before and after to confirm the types changed.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*